In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
from datetime import date
from decimal import Decimal
from uuid import UUID

from pyspark.sql import SparkSession

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, ToolMessage, SystemMessage

from gl import GLSegments

from registry import RegistryClient

from break_analysis.tools import RegistryTools
from break_analysis import BreakAnalysisAgent
from break_analysis.models import BreakRecord

In [3]:
spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('break-agent-dev')
    .config(
        'spark.jars.packages',
        'org.postgresql:postgresql:42.7.7',
    )
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/29 14:40:42 WARN Utils: Your hostname, lionix, resolves to a loopback address: 127.0.1.1; using 192.168.1.7 instead (on interface wlo1)
26/08/29 14:40:42 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/leo/northforge-studio/northforge-finance/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/leo/.ivy2.5.2/cache
The jars for the packages stored in: /home/leo/.ivy2.5.2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-4e5cd3f7-1371-451a-a6c8-22e4b9442a5a;1.0
	confs: [default]
	found org.postgresql#postgresql;42.7.7 in central
	found org.checkerframework#checker-qual;3.49.3 in central
:: resolution report :: resolve 75ms :: artifacts dl 2ms
	:: modules in use:
	org.checkerframework#

In [4]:
llm = ChatOllama(
    model='qwen3:14b-q4_K_M',
    temperature=0
)

In [5]:
registry_client = RegistryClient.from_db(
    spark=spark,
    entity_table='registry.gl_entity',
    department_table='registry.gl_dept',
    branch_table='registry.gl_branch',
    account_table='registry.gl_account',
    sub_account_table='registry.gl_sub_account',
    affiliate_table='registry.gl_affiliate',
    product_table='registry.gl_product',
    book_table='registry.gl_book',
    source_table='registry.gl_source',
)

registry_tools = RegistryTools(registry_client=registry_client)

In [6]:
agent = BreakAnalysisAgent(
    llm=llm,
    registry_tools=registry_tools
)

In [7]:
break_record = BreakRecord(
    recon_result_id=UUID('e5ccd9e5-6760-4e6b-b96f-a717a3b26186'),
    workflow_run_id=UUID('95cd64d3-25a6-480e-8692-58a12fa4a4ad'),
    as_of_date=date(2026, 3, 31),
    segments=GLSegments(
        entity_cd='1000',
        dept_cd='1100',
        branch_cd='NYC',
        gl_account='210000',
        sub_account='2000',
        affiliate_cd='000000',
        product_cd='000000',
        book_cd='LOCAL_GAAP',
        source_cd='NFM_TB',
    ),
    accounted_currency='USD',
    interface_balance=Decimal('-65000.000000000000'),
    gl_balance=Decimal('0E-12'),
    difference_amount=Decimal('-65000.000000000000'),
)

agent.analyze(break_record)

BreakAnalysisResult(recon_result_id=UUID('e5ccd9e5-6760-4e6b-b96f-a717a3b26186'), status=<BreakAnalysisStatus.EXPLAINED: 'EXPLAINED'>, root_cause=<RootCause.REGISTRY_INVALID_SEGMENT: 'REGISTRY_INVALID_SEGMENT'>, explanation="The GL_ACCOUNT segment '210000' was validated as invalid, which explains the reconciliation break.")

In [ ]:
non_break_record = BreakRecord(
    recon_result_id=UUID('e5ccd9e5-6760-4e6b-b96f-a717a3b26186'),
    workflow_run_id=UUID('95cd64d3-25a6-480e-8692-58a12fa4a4ad'),
    as_of_date=date(2026, 3, 31),
    segments=GLSegments(
        entity_cd='1000',
        dept_cd='1100',
        branch_cd='NYC',
        gl_account='410000',
        sub_account='2000',
        affiliate_cd='000000',
        product_cd='000000',
        book_cd='LOCAL_GAAP',
        source_cd='NFM_TB',
    ),
    accounted_currency='USD',
    interface_balance=Decimal('-65000.000000000000'),
    gl_balance=Decimal('0E-12'),
    difference_amount=Decimal('-65000.000000000000'),
)

agent.analyze(non_break_record)

In [ ]:
# spark.stop()